# Running The Alignment Process

This notebook contains a workflow for setting up the pre and post PUNKST Atlas alignment. It is intented to be run on one sample at time. This is not intented to be a full tutorial on how to use any of the specific software used in this workflow. It is simply the minimum information for completeing the task explained above. For more detailed explorations of the included software, tutorial and website links can be found below.

## Software Informtiaon:

- Qupath: 
    - Download: https://qupath.github.io/
    - Tutorials and further info: https://qupath.readthedocs.io/en/stable/docs/tutorials/index.html
- Abba: 
    - Download: https://abba-documentation.readthedocs.io/en/v0.11.0/installation/installation.html
        - This is also the reference page for ABBA. This will contain tutorials and general software information
        - On Windows, the standalone installer is the easiest method if FIJI is not already installed.

- Conda: 
    - Download: https://www.anaconda.com/docs/getting-started/miniconda/install/overview
    - Setting up environment from yaml: https://docs.conda.io/projects/conda/en/latest/user-guide/tasks/manage-environments.html

## Code Setup

In [1]:
import pandas as pd
import geopandas as gpd
import polars as pl
from pathlib import Path
from helpers.extract_region import *

## Step 1 - Creating a Qupath Project

- Start Qupath Application
- Create Project
- Choose an existing folder to store all project files in, or make a new folder
- Name the project file
- Now back in the qupath menu, click add image
- Select Ch000 .tif file
- Ensure the image loads correctly
- File - Save

## Step 2 - Starting in ABBA
- Start the ABBA application
- Seach ABBA in the Fiji search bar and click ABBA Start
- click run
- import the abba project file that was just created ( with all default settings)
- now in the bottom right corner you should see a the name of the slice with colors, click each of the 4 boxes to bring the slice into view

## Step 3 - Aligning the Atlas

- NOTE: This program is very computationally heavy and can be prone to crashes, make sure work is saved often.
- Grab the box above the slice and drag it below the atlas slice that seems closest to the imported slice
- Register -> affline -> interactive transform
    - these setting allow you to get the relative rotation and sizing as close to the atlas slice as possible, it does not have to be perfect but getting it somewhat close will make things easier
- Register -> spline -> Elastix
    - This is an automatic registration step. Play around with matching the different channels and see how it warps.
- Register -> spline -> Big Warp
    - This is the manual registration tool
    - Meant for fine grain adjustments after a few Elastic runs have gotten the sample relatively close.



## Step 4

Export the project:
Export -> QuPath -> Export Qupath Registrations(toggle delete old ROIs)

## Step 5

Grab the GeoJSON files through Qupath:  
- Open the Qupath project file that was created at the start
- go to extensions -> ABBA -> load atlas annotations into open image
    - Make sure you clicked one of the channels files and the left and that there is an actual image on the screen,
        otherwise it will not work and you will get an error
    - Do not split -> change the naming property from ID to atlas_id
- if you can now see that registrations overlayed onto the brain image:
    - File -> Export Objects as GeoJSON -> check pretty JSON -> OK and save the file


# Step 6

In [ ]:
SAMPLE_NAME = "20251013_FT_24hrs_Sag9_ID57476"

PROJ_ROOT = Path.cwd()
LOCAL_OUTPUT = PROJ_ROOT / "local_output"
ALLEN_STRUCTS = PROJ_ROOT / "data" / "allen_structures.json"

REG_GEO = r"S:\Anshutz\Cruz-Martin_Lab\projects\TBI_Project\QP_test_2\aligned_0.geojson"
pixel_size_um = 0.2125

# Crop regions of interest

If FICTURE/PUNKST has already been run, skip this step

In [ ]:


TR_PATH = r"S:\Anshutz\Cruz-Martin_Lab\projects\TBI_Project\data\20251013_FT_24hrs_Sag9_ID57476\Transcripts\transcripts.parquet"
OUTPUT_FIG = LOCAL_OUTPUT / (SAMPLE_NAME + "_cropped_transcript_plot")
CT_OUTPUT = LOCAL_OUTPUT / (SAMPLE_NAME + "_cropped_transcrtipts.tsv")



# Define ROIs by name
target_regions = ['CA1', 'CA2', 'CA3', 'DG-sg', 'DG-mo', 'DG-po', 
                'ProS', 'SUB', 'HATA']

sag9=False
if 'sag9' in str(TR_PATH).lower():
    sag9=True


hid_list = extractAllenInfo(ALLEN_STRUCTS, target_regions)

hippo_data = grabROIData(REG_GEO, hid_list, pixel_size_um, sag9)

cr_df = cropTranscripts(hippo_data, TR_PATH, CT_OUTPUT)

makePlot(OUTPUT_FIG, cr_df, target_regions, hippo_data)

shape: (9, 3)
┌───────────┬─────────┬─────────────────────────────────┐
│ id        ┆ acronym ┆ name                            │
│ ---       ┆ ---     ┆ ---                             │
│ i64       ┆ str     ┆ str                             │
╞═══════════╪═════════╪═════════════════════════════════╡
│ 382       ┆ CA1     ┆ Field CA1                       │
│ 423       ┆ CA2     ┆ Field CA2                       │
│ 463       ┆ CA3     ┆ Field CA3                       │
│ 10703     ┆ DG-mo   ┆ Dentate gyrus, molecular layer  │
│ 10704     ┆ DG-po   ┆ Dentate gyrus, polymorph layer  │
│ 632       ┆ DG-sg   ┆ Dentate gyrus, granule cell la… │
│ 502       ┆ SUB     ┆ Subiculum                       │
│ 484682470 ┆ ProS    ┆ Prosubiculum                    │
│ 589508447 ┆ HATA    ┆ Hippocampo-amygdalar transitio… │
└───────────┴─────────┴─────────────────────────────────┘

Total regions: 9
Kept 11 polygons in correct cluster
Total transcripts in parquet: 328,443,152
Attempting to load t

# Merge with Ficture output

Method 2: The Scripted Export (For Automation)
Since you are building a pipeline, you’ll want to automate this so you don't have to click through menus for every slice. Open the Script Editor in QuPath (Automate > Show Script Editor) and use this snippet:

Since you are automating this, you can use the geopandas library. It is built to handle this exact JSON format (GeoJSON) and perform "Spatial Joins."

Here is the code to map your Ficture transcripts to these specific atlas regions:

In [ ]:

# get ROI DATA containg coordianate info for all hippocampus rubregions, ROIDATA object contains 3 lists corresponding to the polygons, names, & IDs
hid_list = extractAllenInfo(ALLEN_STRUCTS, target_regions)
hippo_data = grabROIData(REG_GEO, hid_list, pixel_size_um, sag9)


